# Does the LLM fall short of a rational client? A paired test

**Exploratory, added after the results (7 Nov 2026). Not part of the pre-registration.**

The pre-registered test (H1) found no clear effect of the band on the LLM client's outcome: B_noelicit − U_noelicit = −0.073, 95% CI [−0.190, +0.038]. On the same scenarios, the rational client gains +0.111.

The rational gain is itself an average over the same 50 scenarios and carries sampling noise. To account for it, and to make the conclusion that the LLM gains less than a rational client more robust, this notebook tests the difference scenario by scenario:

$$\text{gap}_s = \underbrace{[\text{capture}(B^*) - \text{capture}(U^*)]_s}_{\text{rational band effect}} - \underbrace{[\text{capture}(\text{LLM}, B) - \text{capture}(\text{LLM}, U)]_s}_{\text{LLM band effect}}$$

The mean of $\text{gap}_s$ is bootstrapped over scenarios.

Everything below reuses the functions in `disclosure_results.py` unchanged: the same rational benchmark (with the quote rounded to 0.001), the same averaging of replicates within each scenario, and the same bootstrap (10,000 resamples, seed 0).

In [ ]:
import os
from pathlib import Path

# Run from the repository root, wherever the notebook was opened from
ROOT = Path.cwd() if (Path.cwd() / "run.py").exists() else Path.cwd().parent
os.chdir(ROOT)

import numpy as np
import yaml
import matplotlib.pyplot as plt
from disclosure_results import ARMS, load_episodes, benchmark, per_scenario, bootstrap_ci, capture

cal = yaml.safe_load(open("config.yaml"))["calibration"]
g, D = cal["g_points"], cal["disclosure"]
LOGS = {"Main (seeds 1-50)": "logs/main_ornith1.5-35b.jsonl",
        "Held-out (seeds 1001-1020)": "logs/heldout_ornith1.5-35b.jsonl"}

## The test

For each scenario: the rational client's band effect, the LLM's band effect (its replicates averaged), and the gap between them. The p-value is two-sided, from the same bootstrap: twice the share of resampled means on the far side of zero.

In [ ]:
def gap_test(recs, hi, lo, bench):
    seeds = sorted(bench)
    rational = {s: bench[s]["B"].client_capture - bench[s]["U"].client_capture for s in seeds}
    a, b = per_scenario(recs, hi, capture, seeds), per_scenario(recs, lo, capture, seeds)
    common = sorted(set(a) & set(b))
    rat = np.array([rational[s] for s in common])
    llm = np.array([a[s] - b[s] for s in common])
    gap = rat - llm
    mean, ci = bootstrap_ci(gap)
    boot = gap[np.random.default_rng(0).integers(0, len(gap), size=(10_000, len(gap)))].mean(axis=1)
    p = max(2 * min((boot <= 0).mean(), (boot >= 0).mean()), 1 / 10_000)
    return {"n": len(common), "rational": rat, "llm": llm, "gap": gap, "mean": mean, "ci": ci, "p": p,
            "llm_ci": bootstrap_ci(llm)[1]}


results = {}
for name, path in LOGS.items():
    recs = [r for r in load_episodes(path) if r["key"]["condition"] in ARMS]
    bench = benchmark(sorted({r["key"]["seed"] for r in recs}), g, D)
    print(name)
    for hi, lo, label in (("B_noelicit", "U_noelicit", "no elicitation (primary)"), ("B", "U", "with elicitation")):
        res = gap_test(recs, hi, lo, bench)
        results[(name, label)] = res
        print(f"  {label:<25} n {res['n']:>2} | rational {res['rational'].mean():+.3f} | LLM {res['llm'].mean():+.3f} "
              f"| gap {res['mean']:+.3f} [{res['ci'][0]:+.3f}, {res['ci'][1]:+.3f}] | p ~ {res['p']:.4f}")
    print()

Main (seeds 1-50)
  no elicitation (primary)  n 50 | rational +0.111 | LLM -0.073 | gap +0.184 [+0.048, +0.320] | p ~ 0.0082
  with elicitation          n 50 | rational +0.111 | LLM -0.002 | gap +0.113 [-0.027, +0.252] | p ~ 0.1166

Held-out (seeds 1001-1020)
  no elicitation (primary)  n 20 | rational +0.033 | LLM +0.073 | gap -0.040 [-0.270, +0.186] | p ~ 0.7446
  with elicitation          n 20 | rational +0.033 | LLM +0.045 | gap -0.012 [-0.251, +0.237] | p ~ 0.9264



## Check: the informal argument was too generous

The informal argument used only the LLM's interval. The paired gap also carries the noise in the rational benchmark, so its interval should be wider. If the claim survives the wider interval, it survives a stricter test.

In [ ]:
res = results[("Main (seeds 1-50)", "no elicitation (primary)")]
llm_hw = (res["llm_ci"][1] - res["llm_ci"][0]) / 2
gap_hw = (res["ci"][1] - res["ci"][0]) / 2
print(f"LLM band effect alone: half-width {llm_hw:.3f}   (informal argument)")
print(f"Paired gap:            half-width {gap_hw:.3f}   (proper test)")
print(f"Rational band effect across scenarios: sd {res['rational'].std(ddof=1):.3f}; "
      f"correlation with the LLM's effect {np.corrcoef(res['rational'], res['llm'])[0, 1]:+.2f}")

LLM band effect alone: half-width 0.114   (informal argument)
Paired gap:            half-width 0.136   (proper test)
Rational band effect across scenarios: sd 0.314; correlation with the LLM's effect +0.09


## Reading

- **Primary comparison (no elicitation): the LLM falls significantly short of the rational client.** Across the 50 main scenarios, it gains 0.184 less of the surplus from the band, 95% CI [+0.048, +0.320], p ≈ 0.008.
- **With elicitation: the shortfall is not significant.** The gap is +0.113, 95% CI [−0.027, +0.252], p ≈ 0.12.
- **Held-out scenarios: the shortfall cannot be tested.** The rational gain is only +0.033, so there is little to fall short of. The gaps are −0.040 [−0.270, +0.186] without elicitation and −0.012 [−0.251, +0.237] with it.

This test was added after the results, so it is exploratory. It complements H1 rather than replacing it.